# Blobpool heuristic tuning simulator

Interactive exploration of EIP-8070 detection heuristics (H1-H6) against 6 attack profiles.

In [ ]:
import matplotlib.pyplot as plt

from heuristic_sim.blobpool_sim import (
    HeuristicConfig,
    Scenario,
    run_simulation,
)

cfg = HeuristicConfig()
print(f"Includability discount: {cfg.includability_discount}")
print(f"Saturation timeout: {cfg.saturation_timeout}s")
print(f"C_extra max: {cfg.c_extra_max}")
print(f"Max random failure rate: {cfg.max_random_failure_rate}")
print(f"K_high/K_low: {cfg.k_high}/{cfg.k_low}")
print(f"Max request-to-announce ratio: {cfg.max_request_to_announce_ratio}")
print(f"Inbound score discount: {cfg.inbound_score_discount}")
print(f"Provider rate tolerance: {cfg.provider_rate_tolerance}")

In [ ]:
scenario = Scenario(
    n_honest=30,
    attackers=[
        (5, "withholder", {"random_fail_rate": 0.5}),
        (3, "selective_signaler", {"n_senders": 5, "txs_per_sender": 16}),
        (3, "spammer", {"rate": 5.0, "below_includability": True}),
        (2, "spoofer", {}),
        (2, "free_rider", {}),
        (2, "non_announcer", {}),
    ],
    tx_arrival_rate=2.0,
    t_end=300.0,
)
result = run_simulation(cfg, scenario)
print(result.summary_table())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Pool occupancy over time
if result.pool_occupancy:
    ts, counts = zip(*result.pool_occupancy, strict=True)
    axes[0, 0].plot(ts, counts)
    axes[0, 0].set_title("Pool occupancy")
    axes[0, 0].set_xlabel("Time (s)")
    axes[0, 0].set_ylabel("Tx count")

# Disconnects over time
disconnect_times: dict[str, list[float]] = {}
for entry in result.log:
    if entry["event"] == "disconnect":
        b = entry["behavior"]
        disconnect_times.setdefault(b, []).append(entry["t"])

for behavior, times in disconnect_times.items():
    axes[0, 1].hist(times, bins=20, alpha=0.7, label=behavior)
axes[0, 1].set_title("Disconnect timing by behavior")
axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].legend()

# H1 rejections over time
h1_times = [e["t"] for e in result.log if e["event"] == "reject_h1"]
if h1_times:
    axes[1, 0].hist(h1_times, bins=30)
    axes[1, 0].set_title("H1 rejections over time")
    axes[1, 0].set_xlabel("Time (s)")

# H2 evictions over time
h2_times = [e["t"] for e in result.log if e["event"] == "evict_h2_saturation"]
if h2_times:
    axes[1, 1].hist(h2_times, bins=30)
    axes[1, 1].set_title("H2 saturation evictions over time")
    axes[1, 1].set_xlabel("Time (s)")

plt.tight_layout()
plt.show()

## H5 request ratio and bandwidth analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# H5 disconnect timing
h5_times = [e["t"] for e in result.log if e["event"] == "disconnect" and e["reason"] == "h5_request_ratio"]
if h5_times:
    axes[0].hist(h5_times, bins=20, color="purple", alpha=0.7)
    axes[0].set_title(f"H5 disconnect timing (n={len(h5_times)})")
else:
    axes[0].text(0.5, 0.5, "No H5 disconnects", ha="center", va="center", transform=axes[0].transAxes)
    axes[0].set_title("H5 disconnect timing")
axes[0].set_xlabel("Time (s)")

# Request ratio distribution by behavior
from collections import defaultdict
req_ratios: dict[str, list[float]] = defaultdict(list)
for entry in result.log:
    if entry["event"] == "inbound_request_tracked":
        pid = entry["peer_id"]
        # Count requests for this peer
for entry in result.log:
    if entry["event"] == "disconnect":
        pid = entry["peer_id"]
        beh = entry["behavior"]
        req_count = sum(1 for e in result.log if e["event"] == "inbound_request_tracked" and e["peer_id"] == pid)
        ann_count = sum(1 for e in result.log if e["event"] == "accept" and e["peer_id"] == pid)
        ratio = req_count / max(ann_count, 1)
        req_ratios[beh].append(ratio)

if req_ratios:
    behaviors = sorted(req_ratios.keys())
    data = [req_ratios[b] for b in behaviors]
    axes[1].boxplot(data, labels=behaviors)
    axes[1].set_title("Request/announce ratio of disconnected peers")
    axes[1].set_ylabel("Ratio")
    axes[1].tick_params(axis="x", rotation=30)

# Bandwidth by behavior
if result.bandwidth_by_behavior:
    behaviors = sorted(result.bandwidth_by_behavior.keys())
    bw_in = [result.bandwidth_by_behavior[b]["in"] / 1024 for b in behaviors]
    bw_out = [result.bandwidth_by_behavior[b]["out"] / 1024 for b in behaviors]
    x = range(len(behaviors))
    width = 0.35
    axes[2].bar([i - width/2 for i in x], bw_in, width, label="Inbound", alpha=0.8)
    axes[2].bar([i + width/2 for i in x], bw_out, width, label="Outbound", alpha=0.8)
    axes[2].set_xticks(list(x))
    axes[2].set_xticklabels(behaviors, rotation=30)
    axes[2].set_title("Bandwidth by behavior")
    axes[2].set_ylabel("KB")
    axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Sweep saturation_timeout
timeouts = [10, 20, 30, 45, 60, 90]
h2_evictions = []
false_positives = []
for timeout in timeouts:
    cfg_t = HeuristicConfig(saturation_timeout=timeout)
    r = run_simulation(cfg_t, scenario)
    h2_evictions.append(r.h2_evictions)
    false_positives.append(r.false_positives)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(timeouts, h2_evictions, "b-o", label="H2 evictions (attacks caught)")
ax1.set_xlabel("Saturation timeout (s)")
ax1.set_ylabel("H2 evictions", color="b")
ax2 = ax1.twinx()
ax2.plot(timeouts, false_positives, "r-s", label="False positives")
ax2.set_ylabel("False positives", color="r")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("Saturation timeout sensitivity")
plt.show()

In [ ]:
# Sweep max_random_failure_rate
rates = [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
withholder_detected = []
fp_rates = []
for rate in rates:
    cfg_r = HeuristicConfig(max_random_failure_rate=rate)
    r = run_simulation(cfg_r, scenario)
    withholder_detected.append(r.disconnects_by_behavior.get("withholder", 0))
    fp_rates.append(r.false_positives)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(rates, withholder_detected, "b-o", label="Withholders detected")
ax1.set_xlabel("Max random failure rate threshold")
ax1.set_ylabel("Withholders detected", color="b")
ax2 = ax1.twinx()
ax2.plot(rates, fp_rates, "r-s", label="False positives")
ax2.set_ylabel("False positives", color="r")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("Random failure rate threshold sensitivity")
plt.show()

In [ ]:
# Sweep max_request_to_announce_ratio
ratios = [2.0, 3.0, 5.0, 8.0, 10.0, 20.0]
h5_detected = []
fp_ratios = []
for ratio in ratios:
    cfg_r = HeuristicConfig(max_request_to_announce_ratio=ratio)
    r = run_simulation(cfg_r, scenario)
    h5_detected.append(r.h5_disconnects)
    fp_ratios.append(r.false_positives)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(ratios, h5_detected, "b-o", label="H5 disconnects")
ax1.set_xlabel("Max request-to-announce ratio")
ax1.set_ylabel("H5 disconnects", color="b")
ax2 = ax1.twinx()
ax2.plot(ratios, fp_ratios, "r-s", label="False positives")
ax2.set_ylabel("False positives", color="r")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.title("Request-to-announce ratio threshold sensitivity")
plt.show()